# A1 Flight Route Planner — demo

**Does informed search actually pay for itself on a real network?**

This notebook is the demo for CMPE 180A project A1. It runs the delivered
system end to end on the world airline network — 3,387 airports and 66,332
routes — and shows the three things the project claims:

1. Three query modes, from one interface, on one pinned dataset
2. A\* returns **exactly** Dijkstra's answer while expanding far fewer airports
3. The answer is checked against an independent implementation, not just
   against our own tests

## Two rules this notebook follows

**Every measured number is read from a committed `results.json`, never
recomputed here.** A demo that re-runs a timing loop on stage is a demo that
waits, and a number produced live cannot be compared against the report. The
experiments that produced them are in `experiments/`, and
`tests/experiments/` re-derives their answers on every test run.

**The data is pinned and verified.** Opening a snapshot re-hashes every file
against its manifest and refuses to proceed on a mismatch — so a demo that
would have shown the wrong data fails loudly instead of quietly.

What is *computed* live below is only what is fast and deterministic: the
route queries themselves, and the boundary cases.

> **Nothing here is stored output.** The notebook is committed with empty
> cells on purpose; run it top to bottom.

## Setup

Only this first cell differs between Colab and a local checkout. Locally,
`uv sync --group notebooks` (or `make notebook`) has already installed
everything.

In [ ]:
# In Colab, clone the repository and install the package plus the two
# mapping libraries:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima
#   %pip install -q folium pyproj
# Then open this notebook from the clone.
import json
import sys
from pathlib import Path

try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit("install the package first -- see the comment above") from error

# This notebook lives in `notebooks/`, so the repository root is one up. The
# demo modules are not part of the wheel: they sit outside it deliberately, so
# that running them exercises the package's public re-exports the same way a
# user would.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src" / "demos"))

EXPERIMENTS = REPO / "experiments"


def recorded(slug):
    """Return one experiment's committed answer."""
    return json.loads((EXPERIMENTS / slug / "results.json").read_text())


print(f"repository: {REPO}")

## 1. The data, and why it is pinned

`data/processed/` is build output — `make flight_network` overwrites it and the
filenames do not change. A snapshot is an immutable, content-addressed copy:
its id is a hash of its contents, so it can be superseded but never edited.

`Snapshot.open()` re-hashes every file before returning a row.

In [ ]:
from flight_planner.experiments import Snapshot
from route_query_example import SNAPSHOTS_DIR, SNAPSHOT_ID

# Opening it is the verification: every file is re-hashed against the
# manifest before a row is returned.
snapshot = Snapshot.open(SNAPSHOTS_DIR / SNAPSHOT_ID)
catalog = snapshot.catalog()

print(f"snapshot   {snapshot.snapshot_id}")
print(f"criteria   {snapshot.criteria or '(none -- the whole world)'}")
print(f"airports   {len(catalog.airports):,}")
print(f"routes     {len(catalog.routes):,}")
print()
print("Every file above was re-hashed against the manifest on the way in.")

## 2. One question, three different right answers

`FlightPlanner` is handed the algorithm rather than containing one, so the
same call answers three different questions. `HNL → BDL` (Honolulu to
Hartford) is the clearest case.

Watch the **unit** column. BFS's cost is a hop count; the other two are
kilometres. They are never summed, and "BFS won" never means "BFS found a
shorter route".

In [ ]:
from route_query_example import compare_modes, format_result

planner = catalog.planner()
ORIGIN, DESTINATION = "HNL", "BDL"

print(f"{ORIGIN} -> {DESTINATION}\n")
print(f"{'mode':<24}{'cost':>20} {'unit':<5} {'expanded':>14}  route")
results = compare_modes(planner, ORIGIN, DESTINATION)
for name, result in results.items():
    print(f"{name:<24}{format_result(ORIGIN, result)}")

Three things to read off that, and they are the whole project in miniature.

**BFS disagrees because it was asked something else.** Its two-leg itinerary
is 8,616.2 km against Dijkstra's 8,071.5 — so skipping one stop costs 544.7 km.

**A\* agrees with Dijkstra exactly**, not approximately. The assertion below
is on the raw floats, not a rounded display.

**A\* paid about 100× less** to reach the same answer.

In [ ]:
dijkstra = results["shortest distance"]
astar = results["shortest distance (A*)"]

assert astar.cost == dijkstra.cost, "A* and Dijkstra must agree exactly"
assert [leg.flight_number for leg in astar.path] == [
    leg.flight_number for leg in dijkstra.path
], "and on the same itinerary"

print(f"identical to the last bit: {dijkstra.cost!r} == {astar.cost!r}")
print(f"expansions: Dijkstra {dijkstra.nodes_expanded}, "
      f"A* {astar.nodes_expanded} "
      f"({dijkstra.nodes_expanded / astar.nodes_expanded:.0f}x fewer)")

## 3. The measured claim

These are **read, not recomputed** — `experiments/search-cost/` produced them
on the same snapshot, and its `results.json` is committed beside the notebook
that wrote it.

In [ ]:
search_cost = recorded("search-cost")
assert search_cost["snapshot"]["id"] == SNAPSHOT_ID, "measured on other data"

print(f"{'query':<10}{'BFS':>8}{'Dijkstra':>10}{'A*':>6}{'A* saving':>12}"
      f"{'Dijkstra km':>15}{'A* km':>15}")
for row, agree in zip(search_cost["results"]["comparison"],
                      search_cost["results"]["agreement"]):
    expanded = row["expanded"]
    print(f"{row['pair']:<10}{expanded['BFS']:>8,}{expanded['Dijkstra']:>10,}"
          f"{expanded['A*']:>6,}{agree['expansion_ratio']:>11,.0f}x"
          f"{agree['dijkstra_km']:>15,.3f}{agree['astar_km']:>15,.3f}")

gaps = {abs(a["dijkstra_km"] - a["astar_km"])
        for a in search_cost["results"]["agreement"]}
print(f"\nlargest A* vs Dijkstra difference across all five: {max(gaps)} km")

**130× to 784× fewer expansions, and the distances match to the last decimal
place.** That is the project's central claim, measured rather than asserted.

Runtime and the fitted growth exponents come from the same file. The absolute
milliseconds are a property of the machine that recorded them; the exponents
and the node counts are properties of the algorithms and reproduce anywhere.

In [ ]:
environment = search_cost["results"]["environment"]
print(f"recorded on {environment['cpu']}, {environment['cores']} cores, "
      f"Python {environment['python']}\n")

print(f"{'narrowing':<18}{'V + E':>8}{'BFS ms':>9}{'Dijkstra ms':>13}"
      f"{'A* ms':>8}{'A* vs Dij':>11}")
for row in search_cost["results"]["runtime_series"]:
    ms = row["median_ms"]
    print(f"{row['label']:<18}{row['size']:>8,}{ms['BFS']:>9.3f}"
          f"{ms['Dijkstra']:>13.3f}{ms['A*']:>8.3f}"
          f"{ms['Dijkstra'] / ms['A*']:>10.1f}x")

print()
for series, fit in search_cost["results"]["scaling"].items():
    print(f"{series:<12} growth exponent {fit['exponent']:.2f}  "
          f"(r^2 {fit['r_squared']:.3f})")

Only graph construction reaches its O(V + E) exponent. Every *search* comes in
well below its bound, and the fits get worse as the algorithm gets smarter —
because a worst-case bound describes a search that exhausts the graph, and
none of these do. They stop on arrival.

## 4. Where a table stops being enough

Three pairs, from `experiments/route-map/`. The third is the one no column in
the table above can explain.

In [ ]:
route_map_results = recorded("route-map")

print(f"{'pair':<10}{'BFS legs':>10}{'BFS km':>12}{'Dij legs':>10}"
      f"{'Dij km':>12}{'penalty km':>13}")
for pair, row in route_map_results["results"]["comparison"].items():
    bfs, dij = row["BFS"], row["Dijkstra"]
    print(f"{pair:<10}{bfs['legs']:>10}{bfs['km']:>12,.1f}"
          f"{dij['legs']:>10}{dij['km']:>12,.1f}{row['km_penalty']:>13,.1f}")

`SYD-JFK` is the interesting row: **the same two legs either way, 7,057 km
apart.** Hop count cannot explain that, and neither can any column above. The
two answers leave Sydney in opposite directions, which is only visible drawn.

In [ ]:
from flight_planner import BFS, Dijkstra
from flight_planner.viz import RouteMap

# Curved, not straight: a straight line between two coordinates is straight in
# Web Mercator, which is neither the path flown nor the distance the edge
# weight reports. The colours are the same palette the charts use, so Dijkstra
# is the same orange here as in the report's figures.
RouteMap.compare(planner, "HNL", "BDL", {"Dijkstra": Dijkstra(), "BFS": BFS()})

In [ ]:
# The same query, drawn across the antimeridian. The interpolated longitudes
# are carried *past* 180 rather than wrapping, so the line stays continuous --
# and the map is framed on the points actually drawn rather than on the two
# airports, which sit on opposite edges of the world.
RouteMap.compare(planner, "SYD", "JFK", {"Dijkstra": Dijkstra(), "BFS": BFS()})

The two maps above are the only cells that need **network access** —
Leaflet comes from a CDN and the basemap tiles from OpenStreetMap. If the room
has no network, the same three routes are committed as PNGs under
`slides/images/` and appear on the deck's route-map slide, so the finding
survives without the live render.

## 5. How we know the answers are right

Unit tests show the code does what its author expected. Only an **independent
implementation** can show the expectation was right — so every algorithm is
wrapped behind the same interface and run against NetworkX on the same graph.

In [ ]:
parity = recorded("networkx-parity")
print(f"oracle: {parity['results']['oracle']}")
print(f"total cost mismatches: {parity['results']['total_cost_mismatches']}\n")

print(f"{'algorithm':<12}{'pairs':>8}{'cost mismatches':>18}"
      f"{'worst divergence km':>22}{'identical paths':>18}")
for name, row in parity["results"]["parity"].items():
    print(f"{name:<12}{row['pairs_checked']:>8}{row['cost_mismatches']:>18}"
          f"{row['largest_absolute_divergence_km']:>22.2e}"
          f"{row['identical_paths']:>13}/{row['pairs_checked']}")

**Zero cost mismatches across 600 queries.** The worst divergence is 3.6e-12 km
— float summation order, not disagreement.

BFS matching on only 142 of 200 paths is a *finding, not a failure*: when
several routes tie at the same hop count, "fewest stops" does not name one of
them, so two correct implementations may return different ties. Their costs
agree on all 200.

### The boundary cases, computed live

These are cheap, so there is no reason to read them from a file.

In [ ]:
from flight_planner.errors import FlightPlannerError

print("unreachable destination :", planner.find_shortest_route("SPI", "JFK"))
print("origin == destination   :", planner.find_shortest_route("SFO", "SFO"))
try:
    planner.find_shortest_route("ZZZ", "SFO")
except FlightPlannerError as error:
    print(f"unknown airport code    : {type(error).__name__}: {error}")

The third one is the one that matters. An unknown code raises rather than
returning `(inf, [])`, because **"there is no such airport" and "there is no
such route" are different facts** — collapsing them would let a typo look like
a routing result.

The network also supplies its own cyclic and self-loop cases, so those are
tested on real data rather than only on fixtures: `PKN` has a 0.0 km self-loop
sitting beside its genuine departures.

In [ ]:
self_loops = [route for route in catalog.routes
              if route.origin == route.destination]
print(f"self-loops in the delivered network: {len(self_loops)}")
for route in self_loops:
    print(f"  {route.flight_number}  {route.origin.iata_code}"
          f"->{route.destination.iata_code}  {route.distance_km} km")

# Relaxation is strictly-less-than, so arriving again at 0 + 0 is not an
# improvement and the edge is never traversed.
through = planner.search_route("PKN", "SIN", Dijkstra())
print(f"\nPKN -> SIN  {through.cost:,.1f} km in {len(through.path)} legs, "
      f"{through.nodes_expanded} expanded")

## 6. What this does not show

Stated plainly, because a demo that only shows what worked is not evidence.

- **"Cheapest" means shortest distance, not lowest fare.** OpenFlights carries
  no fare data. Nothing here prices a ticket.
- **No schedules, so no connections.** A route is an edge whether or not the
  two flights either side of a stop can actually be connected.
- **Worst-case behaviour is not benchmarked.** Every query above succeeds and
  terminates early; an unreachable destination is the expensive case.
- **One graph shape.** The airline network is small-world — dense hubs, short
  paths. A sparse or grid-like graph would put the heuristic under real
  pressure and these exponents would not carry over.
- **The oracle is independent, not infallible.** Agreement with NetworkX means
  two implementations made the same decisions.

## Where everything lives

| | |
|---|---|
| The report | `make report` → `build/pdf/report.pdf` |
| The deck | `make deck` → generated from the report's own chapters |
| The algorithms | `src/flight_planner/{core,adt,pathfinding,geo}/` — no `heapq`, no `networkx` |
| The measurements | `experiments/*/results.json`, re-derived by `tests/experiments/` |
| This notebook's spine | `src/demos/route_query_example.py` |